In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS medical_project.silver;

In [0]:
enc_df = spark.read.table('medical_project.bronze.encounters')
display(enc_df)

In [0]:
display(enc_df.summary())

In [0]:
#  Remove Fivetran Metadata Columns
from pyspark.sql.functions import col

enc_df = enc_df.select([
    col(c) for c in enc_df.columns if not c.startswith("_")
])


#  Standardize Column Names
import re

def clean_column(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^\w]", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name)
    col_name = col_name.strip("_")
    return col_name

enc_df = enc_df.toDF(*[clean_column(c) for c in enc_df.columns])


#  Convert Data Types
from pyspark.sql.functions import to_timestamp

enc_df = enc_df.withColumn("start", to_timestamp("start")) \
               .withColumn("stop", to_timestamp("stop"))


#  Handle Null Values
enc_df = enc_df.fillna({
    "reasondescription": "unknown",
    "encounter_class": "unknown"
})


# Remove Duplicates
enc_df = enc_df.dropDuplicates(["id"])


# Standardize Categorical Columns
from pyspark.sql.functions import lower, trim

enc_df = enc_df.withColumn("encounter_class", lower(trim(col("encounter_class")))) \
               .withColumn("reasondescription", lower(trim(col("reasondescription"))))


# Filter Invalid Records
enc_df = enc_df.filter(
    col("id").isNotNull() &
    col("patient").isNotNull() &
    col("start").isNotNull()
)


# Calculate Encounter Duration
from pyspark.sql.functions import unix_timestamp, when

# Calculate duration in seconds, then convert to hours
enc_df = enc_df.withColumn(
    "encounter_duration_hours",
    when(
        col("stop").isNotNull(),
        (unix_timestamp("stop") - unix_timestamp("start")) / 3600
    ).otherwise(None)
)

In [0]:
display(enc_df)

In [0]:
# Write to Silver Layer
enc_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.silver.encounters")